In [ ]:
# ===================== RETENTION-QUALITY PIPELINE v2.7 (Project: Retention Cow) =====================
# Inputs: /mnt/data/*.csv (TSV: sep="\t")
# Outputs: ./out_v2_7/
# Focus: User Lifetime (Retention Proxy) & Experience Quality (Risk Weighting)
# -------------------------------------------------------------------------------------------------

import os, re, json, random, logging
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt

# -----------------------------
# Logging
# -----------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("retention_v2_7")

# -----------------------------
# Config
# -----------------------------
DATA_DIR = Path("/mnt/data")
OUT_DIR  = Path("./out_v2_7")
OUT_DIR.mkdir(exist_ok=True)

FILES = {
    "polls_question": "polls_question.csv",
    "polls_questionreport": "polls_questionreport.csv",
    "accounts_userquestionrecord": "accounts_userquestionrecord.csv",
    "accounts_user": "accounts_user.csv",
}

# 가중 위험 지수(Risk_w) 산출을 위한 심각 사유 정의
SEVERE_KEYWORDS = ["선정", "혐오", "불쾌", "괴롭", "성적", "자극", "비하", "모욕", "폭력", "차별"]
SEVERE_WEIGHT = 2.0  # 단순 신고 대비 가중치

# 분석 임계값 및 온보딩 구간
MIN_EXPOSURE_FOR_SEG = 30
IQR_K = 1.5
ONBOARD_DAYS = 7

# 금칙 패턴 (경험 품질 개선)
AVOID_PATTERNS = [
    re.compile(r"(발|땀|입).{0,2}냄새"), re.compile(r"백태"), re.compile(r"키스"),
    re.compile(r"(전남친|여친|남친|애인).*(염탐|몰래|훔쳐|스토킹)"),
    re.compile(r"(SNS).*(염탐|몰래|훔쳐)"), re.compile(r"성별.*(바뀌|바뀐)"),
    re.compile(r"(마스크).*(핵|벗)"), re.compile(r"(등빨|어깨|정수리|콧구멍)"),
    re.compile(r"(우리반|우리학교).*(누구|누가|OOO|ㅇㅇㅇ)")
]

# 질문 생성을 위한 템플릿 풀
ADJ_POOL = ["든든한","믿음직한","웃긴","센스 있는","다정한","힐링되는","대화가 잘 통하는","편한","좋아진"]
STATE_POOL = ["친해지고 싶은","더 알고 싶은","응원하고 싶은"]
VERB_POOL = ["공부하고","밥먹고","게임하고","여행가고","산책하고","운동하고"]
NOUN_POOL = ["팀플","축제","동아리","시험","운동","카페","영화","MT"]

# -----------------------------
# Helpers
# -----------------------------
def read_tsv_safe(path: Path, name: str) -> pd.DataFrame:
    if not path.exists(): raise FileNotFoundError(f"[{name}] 파일 없음: {path}")
    for enc in [None, "utf-8", "utf-8-sig", "cp949"]:
        try:
            sep = "\t" if "csv" in path.name else ","
            return pd.read_csv(path, sep=sep, encoding=enc) if enc else pd.read_csv(path, sep=sep)
        except: continue
    return pd.read_csv(path, sep="\t", engine="python", on_bad_lines="skip")

def to_dt(s): return pd.to_datetime(s, errors="coerce")

def iqr_clip(s: pd.Series, k: float = 1.5):
    s = s.dropna()
    if s.empty: return s
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return s[(s >= q1 - k*iqr) & (s <= q3 + k*iqr)]

# -----------------------------
# 1) Load & Preprocess
# -----------------------------
df_q   = read_tsv_safe(DATA_DIR / FILES["polls_question"], "polls_question")
df_r   = read_tsv_safe(DATA_DIR / FILES["polls_questionreport"], "polls_questionreport")
df_uqr = read_tsv_safe(DATA_DIR / FILES["accounts_userquestionrecord"], "accounts_userquestionrecord")
df_u   = read_tsv_safe(DATA_DIR / FILES["accounts_user"], "accounts_user")

# 타입 및 날짜 정리
for df in [df_q, df_r, df_uqr]:
    if "question_id" not in df.columns and "id" in df.columns: df["question_id"] = df["id"]
    df["question_id"] = pd.to_numeric(df["question_id"], errors="coerce").astype("Int64")

df_uqr["created_at"] = to_dt(df_uqr["created_at"])
df_u["join_at"] = to_dt(df_u["created_at"])
df_u["user_id"] = pd.to_numeric(df_u["id"], errors="coerce").astype("Int64")

# -----------------------------
# 2) Quality Metrics (Experience Quality)
# -----------------------------
df_r["is_severe"] = df_r["reason"].fillna("").apply(lambda x: int(any(k in x for k in SEVERE_KEYWORDS)))
report = df_r.groupby("question_id", as_index=False).agg(
    report_cnt=("id","size"),
    severe_cnt=("is_severe","sum")
)

# -----------------------------
# 3) Retention Proxy (User Lifetime)
# -----------------------------
# 핵심 지표: 24시간 이내 재참여(Return) 여부 측정
uqr_sorted = df_uqr.sort_values(["user_id","created_at"]).copy()
uqr_sorted["next_time"] = uqr_sorted.groupby("user_id")["created_at"].shift(-1)
uqr_sorted["is_retention"] = ((uqr_sorted["next_time"] - uqr_sorted["created_at"]) <= pd.Timedelta(hours=24)).astype(int)

# 질문별 노출 및 전환 지표
df_uqr["is_open"] = (pd.to_numeric(df_uqr["has_read"], errors="coerce").fillna(0) > 0).astype(int)
df_uqr["is_answer"] = (df_uqr["answer_status"].astype(str).str.upper() != "N").astype(int)

retention = uqr_sorted.groupby("question_id", as_index=False).agg(
    exposure_cnt=("id","size"),
    open_cnt=("is_open","sum"),
    answer_cnt=("is_answer","sum"),
    retention_cnt=("is_retention","sum")
)

# -----------------------------
# 4) Join & Calculate Mart
# -----------------------------
metrics_all = df_q[["question_id","question_text"]].merge(retention, on="question_id", how="left")
metrics_all = metrics_all.merge(report, on="question_id", how="left").fillna(0)

# 파생 지표 계산 (보고서)
metrics_all["open_rate"] = np.where(metrics_all["exposure_cnt"]>0, metrics_all["open_cnt"]/metrics_all["exposure_cnt"], 0)
metrics_all["retention_rate"] = np.where(metrics_all["exposure_cnt"]>0, metrics_all["retention_cnt"]/metrics_all["exposure_cnt"], 0)
metrics_all["risk_w"] = np.where(metrics_all["exposure_cnt"]>0,
                                 (metrics_all["report_cnt"] + (SEVERE_WEIGHT-1)*metrics_all["severe_cnt"]) / metrics_all["exposure_cnt"], 0)

# -----------------------------
# 5) Segmentation (Retention-Quality Matrix)
# -----------------------------
seg_base = metrics_all[metrics_all["exposure_cnt"] >= MIN_EXPOSURE_FOR_SEG].copy()
ret_thr = float(iqr_clip(seg_base["retention_rate"]).quantile(0.75)) if not seg_base.empty else 0
risk_thr = float(iqr_clip(seg_base["risk_w"]).quantile(0.75)) if not seg_base.empty else 0

def get_seg(ret, risk):
    if ret >= ret_thr and risk < risk_thr: return "Retention_Cow"   # 고경험품질-고리텐션 (최우선)
    if ret >= ret_thr and risk >= risk_thr: return "Retention_Star" # 고자극-고리텐션 (주의)
    if ret < ret_thr and risk >= risk_thr: return "Retention_Dog"   # 저품질-저리텐션 (퇴출)
    return "Potential_QM"

metrics_all["segment"] = [get_seg(re, ri) for re, ri in zip(metrics_all["retention_rate"], metrics_all["risk_w"])]

# -----------------------------
# 6) Plot (Retention-Risk Matrix)
# -----------------------------

plot_df = metrics_all[metrics_all["exposure_cnt"] >= MIN_EXPOSURE_FOR_SEG].dropna().copy()
if not plot_df.empty:
    plt.figure(figsize=(10, 7))
    for seg in ["Retention_Cow", "Retention_Star", "Retention_Dog", "Potential_QM"]:
        sub = plot_df[plot_df["segment"] == seg]
        plt.scatter(sub["risk_w"], sub["retention_rate"], s=np.log1p(sub["exposure_cnt"])*10, alpha=0.6, label=seg)
    plt.axvline(risk_thr, color="gray", linestyle="--"); plt.axhline(ret_thr, color="gray", linestyle="--")
    plt.title("Retention-Quality Matrix (Project v2.7)"); plt.xlabel("Weighted Risk (Risk_w)"); plt.ylabel("Retention Proxy Rate")
    plt.legend(); plt.tight_layout(); plt.savefig(OUT_DIR / "retention_quality_matrix.png", dpi=180); plt.close()

# -----------------------------
# 7) Offline Generation (Retention Cow Seed)
# -----------------------------
# 리텐션 기여도가 높은 질문을 Seed로 신규 질문 생성
cow_seeds = metrics_all[metrics_all["segment"] == "Retention_Cow"].sort_values("retention_rate", ascending=False).head(50)
seed_list = cow_seeds["question_text"].tolist()

def generate_questions(seeds, n=30):
    # 단순 패턴 추출 및 결합 (보고서 7.2.2 템플릿화 반영)
    results = []
    templates = ["가장 {X} 것 같은 사람은?", "같이 {X} 싶은 사람은?", "요즘 제일 {X} 사람은?", "{X} 하면 떠오르는 사람은?"]
    while len(results) < n:
        tpl = random.choice(templates)
        x = random.choice(ADJ_POOL + STATE_POOL + VERB_POOL + NOUN_POOL)
        q = tpl.replace("{X}", x)
        if not any(p.search(q) for p in AVOID_PATTERNS): results.append(q)
    return list(set(results))

generated_q = generate_questions(seed_list)

# -----------------------------
# 8) Save & Finalize
# -----------------------------
metrics_all.to_csv(OUT_DIR/"question_analysis_full.csv", index=False, encoding="utf-8-sig")
cow_seeds.to_csv(OUT_DIR/"queue_retention_cow.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"generated_questions": generated_q}).to_csv(OUT_DIR/"offline_generated_retention.csv", index=False, encoding="utf-8-sig")

print("\n[Retention_Cow 기반 추천 질문]")
for i, q in enumerate(generated_q[:15], 1): print(f"{i}. {q}")
log.info(f"[DONE] v2.7 Retention-focused pipeline complete -> {OUT_DIR}")